# 02. Exploratory Analysis

## Purpose
With both cohorts cleaned, this notebook explores registration patterns to answer the challenge's two core questions:

1. Where should ad budget go, and on whom?
2. What should change to grow the community and fill Cohort 11?


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

cohort9 = pd.read_csv('../data/processed/cohort9_clean.csv', parse_dates=['Timestamp'])
cohort10 = pd.read_csv('../data/processed/cohort10_clean.csv', parse_dates=['Timestamp'])

print("Cohort 9:", cohort9.shape)
print("Cohort 10:", cohort10.shape)

Cohort 9: (1104, 9)
Cohort 10: (168, 10)


## Registration Channels: Overall Volume

Starting with the raw counts of "How did you hear about our Program?" across both cohorts, before adjusting for the Cohort 9 Jan 22-23 X (Twitter) spike we flagged during cleaning.

In [3]:
channel_col = 'How did you hear about our Program?'

print("Cohort 9 channels:")
print(cohort9[channel_col].value_counts())
print("\nCohort 10 channels:")
print(cohort10[channel_col].value_counts())

Cohort 9 channels:
How did you hear about our Program?
X (Twitter)    692
WhatsApp       197
LinkedIn       169
Referral        27
Facebook        13
Instagram        6
Name: count, dtype: int64

Cohort 10 channels:
How did you hear about our Program?
X (Twitter)           147
Linkedin               11
WhatsApp Community      4
Referral                4
Facebook                2
Name: count, dtype: int64


## Standardizing Channel Names

Channel labels differ slightly between cohorts (e.g. "LinkedIn" vs "Linkedin", "WhatsApp" vs "WhatsApp Community"). Standardizing before any cross-cohort comparison.

In [4]:
channel_map = {
    'x (twitter)': 'X (Twitter)',
    'linkedin': 'LinkedIn',
    'whatsapp': 'WhatsApp',
    'whatsapp community': 'WhatsApp',
    'referral': 'Referral',
    'facebook': 'Facebook',
    'instagram': 'Instagram',
}

cohort9['channel_clean'] = cohort9[channel_col].str.strip().str.lower().map(channel_map)
cohort10['channel_clean'] = cohort10[channel_col].str.strip().str.lower().map(channel_map)

print("Cohort 9 unmapped:", cohort9['channel_clean'].isna().sum())
print("Cohort 10 unmapped:", cohort10['channel_clean'].isna().sum())

print("\nCohort 9 clean channels:")
print(cohort9['channel_clean'].value_counts())
print("\nCohort 10 clean channels:")
print(cohort10['channel_clean'].value_counts())

Cohort 9 unmapped: 0
Cohort 10 unmapped: 0

Cohort 9 clean channels:
channel_clean
X (Twitter)    692
WhatsApp       197
LinkedIn       169
Referral        27
Facebook        13
Instagram        6
Name: count, dtype: int64

Cohort 10 clean channels:
channel_clean
X (Twitter)    147
LinkedIn        11
WhatsApp         4
Referral         4
Facebook         2
Name: count, dtype: int64


## Channel Share: Cohort 9 vs Cohort 10

Comparing channel distribution as a percentage of each cohort's total, since Cohort 9 (1104) is roughly 6.5x the size of Cohort 10 (168). Raw counts alone would overstate Cohort 9's apparent dominance on every channel.

In [5]:
c9_pct = (cohort9['channel_clean'].value_counts(normalize=True) * 100).round(1)
c10_pct = (cohort10['channel_clean'].value_counts(normalize=True) * 100).round(1)

channel_compare = pd.DataFrame({'Cohort 9 %': c9_pct, 'Cohort 10 %': c10_pct}).fillna(0)
channel_compare

,Cohort 9 %,Cohort 10 %
channel_clean,,
Facebook,1.2,1.2
Instagram,0.5,0.0
LinkedIn,15.3,6.5
Referral,2.4,2.4
WhatsApp,17.8,2.4
X (Twitter),62.7,87.5


## Isolating the Jan 22-23 X (Twitter) Spike

Recalculating Cohort 9's channel share with the flagged Jan 22-23 spike rows excluded, to see X (Twitter)'s "steady" organic share versus its spike-inflated share.

In [7]:
spike_mask = (
    (cohort9['Timestamp'].dt.year == 2026) &
    (cohort9['Timestamp'].dt.month == 1) &
    (cohort9['Timestamp'].dt.day.isin([22, 23])) &
    (cohort9['channel_clean'] == 'X (Twitter)')
)

print("Spike rows found:", spike_mask.sum())

cohort9_no_spike = cohort9[~spike_mask]
c9_no_spike_pct = (cohort9_no_spike['channel_clean'].value_counts(normalize=True) * 100).round(1)

pd.DataFrame({'Cohort 9 % (with spike)': c9_pct, 'Cohort 9 % (spike removed)': c9_no_spike_pct, 'Cohort 10 %': c10_pct}).fillna(0)

Spike rows found: 422


,Cohort 9 % (with spike),Cohort 9 % (spike removed),Cohort 10 %
channel_clean,,,
Facebook,1.2,1.9,1.2
Instagram,0.5,0.9,0.0
LinkedIn,15.3,24.8,6.5
Referral,2.4,4.0,2.4
WhatsApp,17.8,28.9,2.4
X (Twitter),62.7,39.6,87.5


## Confirming the Scale of the Jan 22-23 Event

422 X (Twitter) rows (61% of all X (Twitter) registrations) fall in the Jan 22-23 window, far more than the 34 identified earlier (which only covered the missing-name subset). Checking total registration volume across all channels on those two days to confirm this was a cohort-wide event, not isolated to X (Twitter).

In [8]:
full_spike_mask = (
    (cohort9['Timestamp'].dt.year == 2026) &
    (cohort9['Timestamp'].dt.month == 1) &
    (cohort9['Timestamp'].dt.day.isin([22, 23]))
)

print("Total registrations, all channels, Jan 22-23:", full_spike_mask.sum())
print("As % of full Cohort 9:", round(full_spike_mask.sum() / len(cohort9) * 100, 1), "%")
print("\nChannel breakdown within that window:")
print(cohort9[full_spike_mask]['channel_clean'].value_counts())

Total registrations, all channels, Jan 22-23: 462
As % of full Cohort 9: 41.8 %

Channel breakdown within that window:
channel_clean
X (Twitter)    422
WhatsApp        20
LinkedIn         6
Referral         6
Facebook         5
Instagram        3
Name: count, dtype: int64


## Finding: A Single Event Drove 42% of Cohort 9

462 of Cohort 9's 1104 registrations (41.8%) occurred within a 2-day window (Jan 22-23, 2026). Within that window, 91% (422 of 462) came through X (Twitter) specifically, while every other channel saw only a modest bump. This rules out a cohort-wide explanation (like a registration deadline, which would lift all channels roughly evenly) and points to a single X (Twitter)-specific event: a viral post, an influencer mention, a Twitter Space, or similar.

**Implication:** X (Twitter)'s apparent dominance as a channel (62.7% of raw registrations) is largely the result of one short-lived event, not steady organic conversion. With that window removed, X (Twitter) still leads at 39.6%, but LinkedIn (24.8%) and WhatsApp (28.9%) are far stronger steady, repeatable channels than the raw totals suggest. An ad budget built on the raw numbers would overweight a channel that spiked once and underweight the two channels quietly converting people every week.

In [9]:
print("Cohort 9 registration period:", cohort9['Timestamp'].min(), "to", cohort9['Timestamp'].max())

Cohort 9 registration period: 2025-11-06 10:49:00 to 2026-02-04 10:32:00


## Timing Context: Jan 22-23 Was a Late-Cohort Push, Not a Mid-Cohort Anomaly

Cohort 9 ran November 6, 2025 to February 4, 2026 (about 90 days). The Jan 22-23 spike falls roughly 11 days before close, about 85% of the way through the registration window. This is consistent with a deliberate final push to fill remaining seats before the cohort closed, rather than a random mid-cohort event. Whether intentional or not, it demonstrates that concentrated activity on X (Twitter), timed close to a registration deadline, can convert at high volume, a pattern directly relevant to timing recommendations for Cohort 11.

## Course Choice by Channel

Checking which courses each channel brings in, to see whether channels differ in who they attract, not just how many people they attract.

Note: Cohort 9's raw file does not include an Occupation column at all, despite the challenge brief describing an estimated occupation field for Cohort 9. Occupation analysis below uses Cohort 10 only. This is documented as a data limitation, not filled in with a guess.

In [10]:
course_col_9 = 'Choice of Program'
course_col_10 = 'CHOICE OF COURSE'

cohort9['course_clean'] = cohort9[course_col_9].str.strip()
cohort10['course_clean'] = cohort10[course_col_10].str.strip()

print("Cohort 9 courses:")
print(cohort9['course_clean'].value_counts())
print("\nCohort 10 courses:")
print(cohort10['course_clean'].value_counts())

Cohort 9 courses:
course_clean
Data Science and AI              502
Healthcare Data Analytics        272
Financial Analytics              182
Sales and Marketing Analytics    148
Name: count, dtype: int64

Cohort 10 courses:
course_clean
Data science and ML              71
Sales and Marketing Analytics    33
Healthcare Data Analytics        27
Financial Analytics              20
Supply Chain Analytics           11
AI Automation                     6
Name: count, dtype: int64


## Course Catalog Differs Between Cohorts

Course names are internally consistent within each cohort (no formatting issues), but the catalog itself differs between cohorts:

- Cohort 9: Data Science and AI, Healthcare Data Analytics, Financial Analytics, Sales and Marketing Analytics (4 tracks)
- Cohort 10: Data Science and ML, Sales and Marketing Analytics, Healthcare Data Analytics, Financial Analytics, Supply Chain Analytics, AI Automation (6 tracks)

"Data Science and AI" (Cohort 9) and "Data Science and ML" (Cohort 10) are treated as distinct labels rather than merged, since it is not confirmed whether this is a rename of the same track or a genuinely different course. Supply Chain Analytics and AI Automation are new tracks with no Cohort 9 equivalent, so no cohort-over-cohort comparison is possible for these two.

This is a real finding relevant to Question 2: Zion Tech Hub has been expanding its course offerings between cohorts.

In [11]:
pd.crosstab(cohort9['channel_clean'], cohort9['course_clean'], normalize='index').round(3) * 100

course_clean,Data Science and AI,Financial Analytics,Healthcare Data Analytics,Sales and Marketing Analytics
channel_clean,,,,
Facebook,61.5,7.7,23.1,7.7
Instagram,66.7,0.0,0.0,33.3
LinkedIn,36.7,13.0,37.3,13.0
Referral,40.7,11.1,29.6,18.5
WhatsApp,42.6,12.7,33.5,11.2
X (Twitter),48.1,18.9,19.1,13.9


## Finding: Channels Attract Different Course Audiences

Course preference differs meaningfully by channel:

- LinkedIn registrants are nearly twice as likely to choose Healthcare Data Analytics (37.3%) compared to X (Twitter) registrants (19.1%), and correspondingly less likely to choose Data Science and AI (36.7% vs 48.1%).
- X (Twitter) and Instagram skew hardest toward Data Science and AI (48.1% and 66.7% respectively).
- Financial Analytics converts best on X (Twitter) (18.9%) and barely at all on Instagram or Facebook.

**Implication:** Channel selection for ad spend should not be one-size-fits-all. If the goal is filling Data Science and AI seats, X (Twitter) is the stronger bet. If the goal is filling Healthcare Data Analytics specifically, LinkedIn punches above its overall registration share and should not be deprioritized just because its raw volume is lower than X (Twitter)'s.

In [12]:
occ_col = 'Occupation'
cohort10['occupation_clean'] = cohort10[occ_col].str.strip()
print("Unique occupation values:", cohort10['occupation_clean'].nunique())
sorted(cohort10['occupation_clean'].dropna().unique())

Unique occupation values: 112


['A business owner and a corp member',
 'Accountant',
 'Accounting',
 'Administrative Assistant',
 'Administrative Co-ordinator',
 'Administrative professionals',
 'Advocate',
 'Auditor',
 'Baker',
 'Banker',
 'Banking',
 'Bi analysis',
 'Builder',
 'Business man',
 'Businessman',
 'Civil service',
 'Cost controller',
 'Customer Service',
 'Customer Support Specialist',
 'Customer service',
 'Cybersecurity experts',
 'Data Administrator',
 'Data Analyst',
 'Data Officer',
 'Data analyst',
 'Delivery',
 'Dentist',
 'Digital Factory Specialist',
 'Doctor',
 'ENTREPRENEUR',
 'Economist',
 'Educator',
 'Engineer',
 'Engineering Manager',
 'Enterpreneur',
 'Entrepreneur',
 'Entrepreneur / Crypto trader',
 'Event Planner',
 'Fashion designer',
 'Financial analyst',
 'Fleet and Logistics Supervisor',
 'Freelancer',
 'Gadget dealer',
 'General Worker',
 'Graduate',
 'Graduated',
 'HR Officer',
 'Healthcare professional',
 'IT',
 'IT Service desk Support',
 'IT Specialist',
 'Intern',
 'It Supp

## Occupation: Category Mapping

168 registrants produced over 100 distinct free-text occupation values, most appearing only once or twice. Individual raw values are too sparse for meaningful pattern analysis, so occupations are grouped into broader categories. Locations ("Kenya", "Nairobi") and non-answers ("Nil", "No") typed into this field are labeled Invalid Entry, the same treatment used for the equivalent mistake in the Country field.

In [13]:
occupation_map = {
    'accountant': 'Finance/Accounting', 'accounting': 'Finance/Accounting',
    'auditor': 'Finance/Accounting', 'banker': 'Finance/Accounting', 'banking': 'Finance/Accounting',
    'cost controller': 'Finance/Accounting', 'economist': 'Finance/Accounting',
    'financial analyst': 'Finance/Accounting', 'microfinance branch manager': 'Finance/Accounting',
    'risk and compliance': 'Finance/Accounting', 'underwriter': 'Finance/Accounting',

    'bi analysis': 'Tech/Data', 'cybersecurity experts': 'Tech/Data', 'data administrator': 'Tech/Data',
    'data analyst': 'Tech/Data', 'data officer': 'Tech/Data', 'digital factory specialist': 'Tech/Data',
    'it': 'Tech/Data', 'it service desk support': 'Tech/Data', 'it specialist': 'Tech/Data',
    'it support': 'Tech/Data', 'media data analyst': 'Tech/Data', 'quality analyst': 'Tech/Data',
    'software dev': 'Tech/Data', 'software developer': 'Tech/Data', 'statistician': 'Tech/Data',
    'product development': 'Tech/Data',

    'a business owner and a corp member': 'Business/Entrepreneurship', 'business man': 'Business/Entrepreneurship',
    'businessman': 'Business/Entrepreneurship', 'entrepreneur': 'Business/Entrepreneurship',
    'enterpreneur': 'Business/Entrepreneurship', 'entrepreneur / crypto trader': 'Business/Entrepreneurship',
    'freelancer': 'Business/Entrepreneurship', 'gadget dealer': 'Business/Entrepreneurship',
    'real estate agent': 'Business/Entrepreneurship', 'real estate': 'Business/Entrepreneurship',
    'self employed': 'Business/Entrepreneurship', 'trader': 'Business/Entrepreneurship',

    'dentist': 'Healthcare', 'doctor': 'Healthcare', 'healthcare professional': 'Healthcare',
    'medical laboratory scientist (corp member)': 'Healthcare', 'nurse': 'Healthcare',
    'nursing': 'Healthcare', 'pharmacist': 'Healthcare', 'carer': 'Healthcare',

    'administrative assistant': 'Administrative/Management', 'administrative co-ordinator': 'Administrative/Management',
    'administrative professionals': 'Administrative/Management', 'civil service': 'Administrative/Management',
    'civil servant': 'Administrative/Management', 'fleet and logistics supervisor': 'Administrative/Management',
    'hr officer': 'Administrative/Management', 'manager': 'Administrative/Management',
    'procurement manager': 'Administrative/Management', 'project manager': 'Administrative/Management',
    'quality officer': 'Administrative/Management', 'regional administrate': 'Administrative/Management',
    'resource mobilisation officer': 'Administrative/Management', 'security manager': 'Administrative/Management',

    'customer service': 'Sales/Marketing/Customer Service', 'customer support specialist': 'Sales/Marketing/Customer Service',
    'marketing': 'Sales/Marketing/Customer Service', "sale's representative": 'Sales/Marketing/Customer Service',
    'sales executive': 'Sales/Marketing/Customer Service', 'sales/ customer experience': 'Sales/Marketing/Customer Service',
    'relationship manager': 'Sales/Marketing/Customer Service', 'technical pre-sales consultant': 'Sales/Marketing/Customer Service',

    'educator': 'Education', 'lecturer': 'Education', 'lecturing': 'Education', 'teacher': 'Education',

    'engineer': 'Engineering', 'engineering manager': 'Engineering', 'mechanical engineer': 'Engineering',
    'mechatronics engineering': 'Engineering', 'quantity surveyor': 'Engineering',

    'graduate': 'Student/Graduate', 'graduated': 'Student/Graduate', 'intern': 'Student/Graduate',
    'medical student': 'Student/Graduate', 'student': 'Student/Graduate', 'student/photographer': 'Student/Graduate',

    'baker': 'Other/Skilled Trade', 'builder': 'Other/Skilled Trade', 'delivery': 'Other/Skilled Trade',
    'event planner': 'Other/Skilled Trade', 'fashion designer': 'Other/Skilled Trade',
    'general worker': 'Other/Skilled Trade', 'maritime officer': 'Other/Skilled Trade',
    'mechanic': 'Other/Skilled Trade', 'military': 'Other/Skilled Trade', 'porter': 'Other/Skilled Trade',
    'sommelier': 'Other/Skilled Trade', 'travel agent': 'Other/Skilled Trade', 'advocate': 'Legal', 'lawyer': 'Legal',

    'not employed': 'Unemployed', 'unemployed': 'Unemployed', 'unemployment': 'Unemployed',

    'kenya': 'Invalid Entry', 'nairobi': 'Invalid Entry', 'nil': 'Invalid Entry', 'no': 'Invalid Entry',
}

cohort10['occupation_category'] = cohort10['occupation_clean'].str.strip().str.lower().map(occupation_map)

print("Unmapped rows:", cohort10['occupation_category'].isna().sum())
cohort10['occupation_category'].value_counts(dropna=False)

Unmapped rows: 1


occupation_category
Student/Graduate                    38
Tech/Data                           22
Business/Entrepreneurship           19
Finance/Accounting                  15
Administrative/Management           14
Other/Skilled Trade                 13
Sales/Marketing/Customer Service    10
Unemployed                          10
Healthcare                           9
Education                            6
Engineering                          5
Invalid Entry                        4
Legal                                2
NaN                                  1
Name: count, dtype: int64

In [14]:
cohort10[cohort10['occupation_category'].isna()][['NAME', 'occupation_clean']]

,NAME,occupation_clean
159,Genesis,NaN


## Occupation Categories: Result

Grouped 100+ raw occupation values into 12 broad categories. The single unmapped row (Genesis) is the same isolated missing Occupation value identified during cleaning, already accounted for and intentionally left blank. Student/Graduate (38, 22.6%) and Tech/Data (22, 13.1%) are the two largest groups, followed by Business/Entrepreneurship (19) and Finance/Accounting (15). 4 rows (2.4%) are Invalid Entry (a location or non-answer typed into the field).

In [15]:
pd.crosstab(cohort10['channel_clean'], cohort10['occupation_category'], normalize='index').round(3) * 100

occupation_category,Administrative/Management,Business/Entrepreneurship,Education,Engineering,Finance/Accounting,Healthcare,Invalid Entry,Legal,Other/Skilled Trade,Sales/Marketing/Customer Service,Student/Graduate,Tech/Data,Unemployed
channel_clean,,,,,,,,,,,,,
Facebook,0.0,0.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,50.0
LinkedIn,9.1,0.0,0.0,0.0,0.0,9.1,0.0,0.0,0.0,0.0,45.5,36.4,0.0
Referral,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25.0,50.0,25.0,0.0
WhatsApp,25.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25.0,50.0,0.0
X (Twitter),8.2,13.0,4.1,3.4,9.6,5.5,2.7,1.4,8.9,6.2,20.5,10.3,6.2


## Occupation by Channel: Sample Size Caveat

The channel-by-occupation percentages must be read against each channel's actual sample size, not taken at face value:

| Channel | n |
|---|---|
| X (Twitter) | 147 |
| LinkedIn | 11 |
| WhatsApp | 4 |
| Referral | 4 |
| Facebook | 2 |

Only X (Twitter) has enough volume for its percentage breakdown to be treated as a real pattern. LinkedIn's numbers are directional at best. Facebook, WhatsApp, and Referral's percentages (based on 2-4 people each) are not meaningful and should not be used to justify any occupation-targeting decision.

**What the data reliably shows:** Within X (Twitter) (n=147), Student/Graduate is the largest group (20.5%), followed by Business/Entrepreneurship (13.0%), Tech/Data (10.3%), and Finance/Accounting (9.6%). No single occupation dominates X (Twitter); it pulls a broad, mixed audience rather than one specific professional type.

## Registration Timing

Looking at when people register: day of week, and volume over the course of each cohort's registration window. This feeds both Q1 (timing of ad spend) and Q2 (timing recommendations for Cohort 11).

In [16]:
cohort9['day_of_week'] = cohort9['Timestamp'].dt.day_name()
cohort10['day_of_week'] = cohort10['Timestamp'].dt.day_name()

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

c9_dow = cohort9['day_of_week'].value_counts(normalize=True).reindex(day_order) * 100
c10_dow = cohort10['day_of_week'].value_counts(normalize=True).reindex(day_order) * 100

pd.DataFrame({'Cohort 9 %': c9_dow.round(1), 'Cohort 10 %': c10_dow.round(1)})

,Cohort 9 %,Cohort 10 %
day_of_week,,
Monday,18.3,2.4
Tuesday,11.7,0.6
Wednesday,12.0,4.8
Thursday,29.9,58.9
Friday,19.8,28.6
Saturday,6.2,4.8
Sunday,2.1,NaN


## Investigating the Thursday/Friday Concentration

Cohort 10 shows an unusually extreme concentration: 87.5% of registrations fall on Thursday or Friday, compared to Cohort 9's milder 49.7% on the same two days. Checking whether this reflects a recurring weekly promotional pattern (e.g. content posted the same day each week) rather than organic registrant behavior.

In [17]:
print("Cohort 10 registration period:", cohort10['Timestamp'].min(), "to", cohort10['Timestamp'].max())

# Week-by-week volume to see if Thursday spikes repeat weekly or come from one or two big days
cohort10['week'] = cohort10['Timestamp'].dt.isocalendar().week
weekly_thu_fri = cohort10[cohort10['day_of_week'].isin(['Thursday', 'Friday'])].groupby(['week', 'day_of_week']).size().unstack(fill_value=0)
weekly_thu_fri

Cohort 10 registration period: 2026-05-06 11:03:32 to 2026-06-04 05:33:32


day_of_week,Friday,Thursday
week,,
19,1,3
20,41,84
21,1,3
22,5,8
23,0,1


## Finding: Cohort 10 Is Dominated by a Single 2-Day Event

Week-by-week breakdown shows 125 of 168 Cohort 10 registrations (74.4%) occurred on a single Thursday/Friday (May 14-15, 2026), ISO week 20. Every other week in the registration period combined accounts for only 43 registrations. This is not a recurring weekly pattern, it is one concentrated event, structurally similar to the Jan 22-23 spike found in Cohort 9, but proportionally even larger.

In [18]:
may_14_15 = cohort10[(cohort10['Timestamp'].dt.date >= pd.Timestamp('2026-05-14').date()) &
                       (cohort10['Timestamp'].dt.date <= pd.Timestamp('2026-05-15').date())]

print("Rows in May 14-15 window:", len(may_14_15))
print("\nChannel breakdown:")
print(may_14_15['channel_clean'].value_counts())

Rows in May 14-15 window: 125

Channel breakdown:
channel_clean
X (Twitter)    120
Facebook         2
WhatsApp         1
LinkedIn         1
Referral         1
Name: count, dtype: int64


## Recalculating Cohort 10 Channel Share With the Spike Removed

In [19]:
c10_spike_mask = (cohort10['Timestamp'].dt.date >= pd.Timestamp('2026-05-14').date()) & \
                  (cohort10['Timestamp'].dt.date <= pd.Timestamp('2026-05-15').date())

cohort10_no_spike = cohort10[~c10_spike_mask]
c10_no_spike_pct = (cohort10_no_spike['channel_clean'].value_counts(normalize=True) * 100).round(1)

print("Cohort 10 rows remaining after removing spike:", len(cohort10_no_spike))

pd.DataFrame({
    'Cohort 9 % (spike removed)': c9_no_spike_pct,
    'Cohort 10 % (spike removed)': c10_no_spike_pct
}).fillna(0)

Cohort 10 rows remaining after removing spike: 43


,Cohort 9 % (spike removed),Cohort 10 % (spike removed)
channel_clean,,
Facebook,1.9,0.0
Instagram,0.9,0.0
LinkedIn,24.8,23.3
Referral,4.0,7.0
WhatsApp,28.9,7.0
X (Twitter),39.6,62.8


## Finding: Steady Channel Performance, Spikes Removed

With both cohorts' one-off spike events removed, a clearer channel story emerges:

- **X (Twitter) is genuinely growing as a steady channel**, not just spike-driven: 39.6% (Cohort 9) to 62.8% (Cohort 10).
- **LinkedIn is the most stable, predictable channel**: 24.8% to 23.3%, essentially unchanged.
- **WhatsApp is fading**: 28.9% to 7.0%, the steepest decline of any channel.
- **Facebook and Instagram have effectively disappeared**: present in Cohort 9 (1.9% and 0.9%), absent entirely in Cohort 10.

Caveat: Cohort 10's spike-removed sample is small (n=43 vs Cohort 9's n=682), so its exact percentages are less statistically stable. The direction of these trends (X (Twitter) growing, WhatsApp declining) is consistent enough to trust; the precise magnitude should be treated as indicative.